In [4]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

import multiprocessing


In [5]:
# Download associaiton file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/regenie_127phenotypes_lofteeHC_mac20.parquet -o /home/dnanexus/data_dir/
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/loftee_mac20_associations_bh_corrected.parquet -o /home/dnanexus/data_dir/

bt = pl.read_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20.parquet')
bt = bt.filter(pl.col('pval_fdr')<=0.05)
bt

dxpy.utils.resolver.ResolutionError: Unable to resolve "regenie_127phenotypes_lofteeHC_mac20.parquet" to a data object or folder name in '/processed_data/REGENIE_results'


region,chr,phenotype,cohort,model,effect,lci_effect,uci_effect,pval,aaf,num_cases,cases_ref,cases_het,cases_alt,num_controls,controls_ref,controls_het,controls_alt,info,pval_fdr,pval_bonf
str,i64,str,str,str,f64,f64,f64,f64,f64,i64,i64,i64,i64,str,str,str,str,str,f64,f64
"""ENSG00000132855""",1,"""apolipoprotein_a_int""","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.46207,-0.612266,-0.311875,1.6422e-9,0.000166,386584,386456,128,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.076632;MAC=128.00…",0.000007,0.003328
"""ENSG00000052841""",11,"""apolipoprotein_a_int""","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.807306,-1.19208,-0.422532,0.000039,0.000049,386584,386509,75,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.196317;MAC=38.000…",0.038502,1.0
"""ENSG00000110243""",11,"""apolipoprotein_a_int""","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.449444,-0.633744,-0.265144,0.000002,0.00011,386584,386499,85,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.094032;MAC=85.001…",0.003376,1.0
"""ENSG00000118137""",11,"""apolipoprotein_a_int""","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-2.60164,-2.94144,-2.26184,6.6771e-51,0.000032,386584,386559,25,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.173370;MAC=25.000…",4.5099e-46,1.3530e-44
"""ENSG00000173064""",12,"""apolipoprotein_a_int""","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.873767,-1.24455,-0.502986,0.000004,0.000053,386584,386503,81,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.189177;MAC=41.000…",0.006545,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000182095""",7,"""forced_expiratory_volume_in_1s…","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.860502,-1.2525,-0.468501,0.000017,0.00003,405948,405899,49,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.200004;MAC=24.500…",0.02095,1.0
"""ENSG00000164741""",8,"""forced_expiratory_volume_in_1s…","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.642983,-0.947885,-0.338081,0.000036,0.00005,405948,405867,81,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.155565;MAC=40.500…",0.036208,1.0
"""ENSG00000205189""",8,"""forced_expiratory_volume_in_1s…","""scores_bgen_lofteeHC_mac20""","""ADD-WGR-LR""",-0.635227,-0.942006,-0.328447,0.000049,0.000025,405948,405928,20,0,"""NA""","""NA""","""NA""","""NA""","""REGENIE_SE=0.156523;MAC=20.000…",0.045581,1.0


In [ ]:
# Download annotation file for variant-gene mapping
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fill_na.parquet -o /home/dnanexus/data_dir/

anno = pl.read_parquet(
    '/home/dnanexus/data_dir/annotations_fill_na.parquet', 
    columns=['id', 'region']
)

anno

[===========================================================>] Completed 10,077,386,508 of 10,077,386,508 bytes (100%) /home/dnanexus/data_dir/annotations_fill_na.parquett


id,region
str,str
"""chr18:56603287:T:G""","""ENSG00000091164"""
"""chr18:56611517:A:G""","""ENSG00000091164"""
"""chr19:34340475:G:A""","""ENSG00000166398"""
"""chr19:5256210:T:C""","""ENSG00000105426"""
"""chr19:5257316:G:C""","""ENSG00000105426"""
…,…
"""chr16:179290:T:A""","""ENSG00000086506"""
"""chr7:148348869:A:G""","""ENSG00000174469"""
"""chr5:138680487:A:G""","""ENSG00000044115"""


In [8]:
anno_pheno = (
    anno.join(
        bt.select(['region', 'phenotype']).unique(), 
        on='region', 
        how='inner'
    )
)

anno_pheno

id,region,phenotype
str,str,str
"""chr19:34340475:G:A""","""ENSG00000166398""","""seated_height_int"""
"""chr19:34340475:G:A""","""ENSG00000166398""","""sitting_height_int"""
"""chr2:215390704:C:T""","""ENSG00000115414""","""forced_expiratory_volume_in_1s…"
"""chr2:178702309:A:G""","""ENSG00000155657""","""arm_fat_percentage_left_int"""
"""chr2:178702309:A:G""","""ENSG00000155657""","""arm_predicted_mass_right_int"""
…,…,…
"""chr15:72103059:G:GC""","""ENSG00000066933""","""sitting_height_int"""
"""chr17:64662044:G:C""","""ENSG00000108854""","""standing_height_int"""
"""chr17:64662044:G:C""","""ENSG00000108854""","""forced_vital_capacity_fvc_int"""


In [9]:
# Download phenotypes: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o /home/dnanexus/data_dir/

phenos = (
    pl.read_parquet('/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet')
    .select(['individual'] + bt['phenotype'].unique().to_list())
    .rename({'individual':'sample'})
)

phenos

# long_phenos = (
#     phenos
#     .unpivot(
#         index='sample',
#         on=bt['phenotype'].unique().to_list(),
#         variable_name='phenotype',
#         value_name='pheno_value'
#     )
#     .drop_nulls()
# )

# print(long_phenos['phenotype'].value_counts(sort=True))
# long_phenos

Error: path "/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet"
already exists but -f/--overwrite was not set


sample,hand_grip_strength_right_int,reticulocyte_percentage_int,weight_impedance_int,monocyte_count_int,arm_predicted_mass_left_int,mean_platelet_thrombocyte_volume_int,leg_fat_mass_right_int,mothers_age_at_death_int,ldl_direct_int,neutrophill_percentage_int,leg_fatfree_mass_right_int,aspartate_aminotransferase_int,calcium_int,red_blood_cell_erythrocyte_distribution_width_int,birth_weight_int,leg_fatfree_mass_left_int,townsend_deprivation_index_at_recruitment_int,arm_predicted_mass_right_int,apolipoprotein_a_int,leg_predicted_mass_right_int,arm_fat_mass_left_int,heel_bone_mineral_density_bmd_tscore_automated_right_int,cholesterol_int,reticulocyte_count_int,body_mass_index_bmi_int,heel_bone_mineral_density_bmd_left_int,apolipoprotein_b_int,haematocrit_percentage_int,lipoprotein_a_int,trunk_predicted_mass_int,forced_vital_capacity_fvc_int,leg_fat_mass_left_int,arm_fat_mass_right_int,arm_fat_percentage_right_int,age_at_hysterectomy_int,phosphate_int,…,lymphocyte_percentage_int,diastolic_blood_pressure_automated_reading_int,arm_fatfree_mass_left_int,mean_corpuscular_haemoglobin_int,white_blood_cell_leukocyte_count_int,standing_height_int,platelet_distribution_width_int,microalbumin_in_urine_int,creatinine_int,leg_predicted_mass_left_int,gamma_glutamyltransferase_int,vitamin_d_int,haemoglobin_concentration_int,basal_metabolic_rate_int,alkaline_phosphatase_int,neutrophill_count_int,seated_height_int,basophill_percentage_int,triglycerides_int,pulse_rate_automated_reading_int,alanine_aminotransferase_int,creatinine_enzymatic_in_urine_int,trunk_fat_mass_int,heel_bone_mineral_density_bmd_right_int,immature_reticulocyte_fraction_int,arm_fatfree_mass_right_int,mean_corpuscular_volume_int,peak_expiratory_flow_pef_int,lymphocyte_count_int,forced_vital_capacity_fvc_best_measure_int,eosinophill_percentage_int,hdl_cholesterol_int,stroke_volume_during_pwa_int,glycated_haemoglobin_hba1c_int,pulse_rate_int,forced_expiratory_volume_in_1second_fev1_predicted_percentage_int,forced_expiratory_volume_in_1second_fev1_predicted_int
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1000020""",-0.013968,-0.909716,-0.879798,1.497723,-0.690214,-0.255897,-0.874339,-0.615486,1.073916,0.997191,-0.669404,0.390311,-0.216865,0.670406,null,-0.706429,1.293886,-0.745691,-0.873931,-0.701679,-0.923744,null,0.938767,-0.85982,-0.771904,null,0.937826,0.031104,0.341151,-0.770076,0.155806,-0.881454,-0.788784,-0.698221,null,0.428045,…,-1.257792,0.264023,-0.719405,-0.441621,1.08969,-0.560732,-0.752778,null,-0.944843,-0.722047,-0.707955,-0.234295,-0.142614,-0.778486,-0.201615,1.300824,-0.327381,1.129913,1.687425,0.192136,-0.967888,0.049535,-0.918032,null,-0.752744,-0.727854,-0.160414,0.09859,-0.228096,0.237487,-1.531674,-0.903665,null,0.285142,null,0.375105,-0.336233
"""1000107""",0.136971,-0.205456,-0.561876,-1.319146,0.203399,-1.996854,-0.691302,null,0.039055,-0.045546,0.112188,-0.430284,null,-0.111534,0.555677,-0.242606,0.1572,0.238835,null,0.102561,-1.074427,null,0.301096,-0.321747,-0.474691,null,-0.146645,-0.497543,0.590703,0.191941,-0.094969,-0.552037,-1.002036,-0.960104,null,null,…,0.402776,1.268623,0.107505,-0.069626,-1.290805,-0.146177,-0.023053,null,0.025996,-0.28103,-0.669581,0.942069,-0.542065,-0.016443,-1.666109,-1.057355,-0.770639,0.175679,0.205809,1.104342,-0.986873,0.417112,-1.162217,null,-0.84863,0.38869,-0.206012,0.41017,-0.571563,-0.110614,-0.21631,null,null,-0.863708,null,-0.023328,-0.159696
"""1000161""",-0.100593,0.608602,-0.875212,-0.974407,-0.567327,0.150631,-0.399156,-0.093698,0.167881,-0.076697,-0.763564,-1.162813,0.439312,-0.448948,-1.509366,-0.493271,-0.005957,-0.925188,0.360865,-0.651654,-0.353129,-0.904707,0.521059,0.481383,-0.034993,-1.039396,0.007919,0.436301,null,-0.661212,-0.728217,-0.444823,-0.2

In [ ]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
)
long_gt.head().collect()

[===========================================================>] Completed 27,974,638,284 of 27,974,638,284 bytes (100%) /home/dnanexus/data_dir/gt_long.parquett                                                        ] Downloaded 603,979,776 of 27,974,638,284 bytes (2%) /home/dnanexus/data_dir/gt_long.parquet[==========>                                                 ] Downloaded 5,033,164,800 of 27,974,638,284 bytes (17%) /home/dnanexus/data_dir/gt_long.parquet[============================================>               ] Downloaded 21,038,628,864 of 27,974,638,284 bytes (75%) /home/dnanexus/data_dir/gt_long.parquet


id,sample,gt
str,str,i8
"""chr10:20020006:C:T""","""5317620""",1
"""chr10:20020007:TTTTCTTGC:T""","""5546988""",1
"""chr10:20020010:T:C""","""2793793""",1
"""chr10:20020014:G:A""","""1722739""",1
"""chr10:20020014:G:A""","""1139673""",1


In [ ]:
# Get the list of phenotype columns (names) for aggregation
pheno_cols = [c for c in phenos.collect_schema().names() if c != "sample"]

# 2. The Main Pipeline
result = (
    long_gt
    .filter(pl.col("gt") == 1)

    # A. Join Genotypes with ALL Phenotypes at once (Wide)
    #    Row count stays same as long_gt! No explosion.
    .join(
        phenos.lazy(), 
        on="sample", 
        how="inner"
    )
    
    # B. Group by Variant ID
    .group_by("id")
    .agg(
        # C. Compute stats for ALL phenotype columns simultaneously
        #    This is extremely fast in Polars
        pl.col(pheno_cols).count().name.suffix("_n"),
        pl.col(pheno_cols).mean().name.suffix("_mean"),
        pl.col(pheno_cols).std().name.suffix("_std"),
    )
    
    # C. Pack triplets into Structs
    #    We create one column per phenotype, e.g., 'height' = {n: 10, mean: 1.5, std: 0.2}
    .select(
        pl.col("id"),
        *[
            pl.struct(
                n    = pl.col(f"{p}_n"),
                mean = pl.col(f"{p}_mean"),
                std  = pl.col(f"{p}_std")
            ).alias(p) 
            for p in pheno_cols
        ]
    )
    
    # D. Unpivot the Struct columns
    #    This turns columns "height", "bmi" -> rows under column "phenotype"
    #    The values are carried in the "stats" struct.
    .unpivot(
        index="id",
        on=pheno_cols,          # The struct columns we just made
        variable_name="phenotype",
        value_name="stats"
    )
    
    # E. Unnest the stats
    #    Expands the struct {n, mean, std} into three actual columns
    .unnest("stats")

    .sink_parquet('/home/dnanexus/data_dir/loftee_mac20_quant_pheno_assocs_EURunrelated_appv.parquet', engine="streaming")
    # .collect(engine="streaming")
)

result

## Old code

In [15]:
sel_pheno = 'standing_height_int'

tmp = (
    anno_pheno.filter(pl.col('phenotype')==sel_pheno)
    .lazy()
    .join(
        long_gt,
        on='id',
        how='inner'
    )

    .join(
        long_phenos.filter(pl.col('phenotype')==sel_pheno).lazy(),
        on=['sample', 'phenotype'],
        how='inner'
    )

    .group_by(['id', 'region', 'phenotype'])
    .agg(
        n_individuals = pl.len().cast(pl.Int32),
        mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
        std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
    )

    .collect(engine='streaming')
)

tmp

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,str,i32,f32,f32
"""chr11:48078046:GT:G""","""ENSG00000149177""","""standing_height_int""",14,0.203621,0.680058
"""chr11:48070210:G:A""","""ENSG00000149177""","""standing_height_int""",10,0.163916,0.327801
"""chr11:48079038:T:A""","""ENSG00000149177""","""standing_height_int""",1,-0.729936,null
"""chr11:48076801:G:A""","""ENSG00000149177""","""standing_height_int""",1,-1.780711,null
"""chr11:48076294:T:C""","""ENSG00000149177""","""standing_height_int""",1,0.398215,null
…,…,…,…,…,…
"""chr11:14682021:TTACAA:T""","""ENSG00000152270""","""standing_height_int""",2,0.066053,0.076738
"""chr11:47995254:TATC:T""","""ENSG00000149177""","""standing_height_int""",1,0.400572,null
"""chr11:47998675:G:A""","""ENSG00000149177""","""standing_height_int""",7,-0.291961,0.758731
